## Importing necessary libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm

## Import Dataset

In [ ]:
df=pd.read_csv('../data/fashion.csv')

# EDA

In [ ]:
df.head()

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2906 entries, 0 to 2905
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ProductId     2906 non-null   int64
 1   Gender        2906 non-null   str  
 2   Category      2906 non-null   str  
 3   SubCategory   2906 non-null   str  
 4   ProductType   2906 non-null   str  
 5   Colour        2906 non-null   str  
 6   Usage         2906 non-null   str  
 7   ProductTitle  2906 non-null   str  
 8   Image         2906 non-null   str  
 9   ImageURL      2906 non-null   str  
dtypes: int64(1), str(9)
memory usage: 227.2 KB


,ProductId
count,2906.000000
mean,29229.495526
std,15527.390981
min,1636.000000
25%,15256.250000
50%,34016.500000
75%,40769.250000
max,59943.000000


In [ ]:
def get_resnet50_encoder(embedding_dim=512, pretrained=True, train_backbone=False):
    resnet = models.resnet50(pretrained=pretrained)
    modules = list(resnet.children())[:-1]  # Remove last FC layer
    backbone = nn.Sequential(*modules)

    model = nn.Sequential(
        backbone,
        nn.Flatten(),
        nn.Linear(2048, embedding_dim),
        nn.functional.normalize  # L2 normalize embeddings
    )

    # Optionally freeze backbone
    if not train_backbone:
        for param in backbone.parameters():
            param.requires_grad = False

    return model

In [ ]:
# ===== 2. Dataset & DataLoader =====
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
class L2Normalize(nn.Module):
    def __init__(self, p=2, dim=1, eps=1e-12):
        super().__init__()
        self.p = p
        self.dim = dim
        self.eps = eps

    def forward(self, x):
        return nn.functional.normalize(x, p=self.p, dim=self.dim, eps=self.eps)

In [ ]:
import os
import shutil

# Path to the .ipynb_checkpoints directory that might be causing issues
checkpoint_dir_train = "./sample_data/train/.ipynb_checkpoints"
checkpoint_dir_val = "data/val/.ipynb_checkpoints"

# Check if the directory exists and remove it
if os.path.exists(checkpoint_dir_train) and os.path.isdir(checkpoint_dir_train):
    shutil.rmtree(checkpoint_dir_train)
    print(f"Removed problematic directory: {checkpoint_dir_train}")

if os.path.exists(checkpoint_dir_val) and os.path.isdir(checkpoint_dir_val):
    shutil.rmtree(checkpoint_dir_val)
    print(f"Removed problematic directory: {checkpoint_dir_val}")

train_dataset = datasets.ImageFolder("./sample_data/train", transform=transform)
val_dataset = datasets.ImageFolder("./sample_data/train/val", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

# ===== 3. Model, Loss, Optimizer =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_resnet50_encoder(embedding_dim=256, pretrained=True, train_backbone=True).to(device)

# Example: contrastive learning loss (InfoNCE, Triplet, etc.)
# Here we use TripletMarginLoss for demonstration
criterion = nn.TripletMarginLoss(margin=1.0, p=2)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        # For Triplet loss, you need (anchor, positive, negative) samples
        # Here we assume you have a custom dataset that returns them
        # This is just a placeholder
        anchor, positive, negative = batch  # Replace with your triplet dataset
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

        optimizer.zero_grad()
        emb_a = model(anchor)
        emb_p = model(positive)
        emb_n = model(negative)

        loss = criterion(emb_a, emb_p, emb_n)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating"):
            anchor, positive, negative = batch
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

            emb_a = model(anchor)
            emb_p = model(positive)
            emb_n = model(negative)

            loss = criterion(emb_a, emb_p, emb_n)
            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
EPOCHS = 10
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss = validate_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")

# ===== 7. Save the trained encoder =====
torch.save(model.state_dict(), "resnet50_encoder.pth")
print("Model saved as resnet50_encoder.pth")